<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/linearRegressionGoldnIDR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
!pip install ydata-profiling
from ydata_profiling import ProfileReport
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
TROY_OUNCE_TO_GRAM = 31.1035   # 1 troy ounce = 31.1035 gram
pd.set_option('display.max_columns', lambda x: '%.2f' % x)
ticker_map = {
    "GC=F"    : "gold",
    "USDIDR=X": "usdidr",
}

START = "2023-01-01"
END   = datetime.today().strftime("%Y-%m-%d")

LAG_STEPS       = [1, 3, 7]
ROLLING_WINDOWS = [3, 7]

In [ ]:
raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True
)

df = raw["Close"].rename(columns=ticker_map).add_suffix("_close")
print(f"\nShape setelah download: {df.shape}")
print(df)
print(f"Rentang tanggal awal  : {df.index.min().date()} s.d. {df.index.max().date()}")

In [ ]:
# TAHAP 2: PENYELARASAN TIMEZONE (SHIFT +1 HARI)
GLOBAL_TICKERS = list(ticker_map.keys())

for ticker_name in GLOBAL_TICKERS:
  col = f"{ticker_name}_close"
  if col in df.columns:
    df[col] = df[col].shift(1)

In [ ]:
# TAHAP 3: REINDEX KE KALENDER HARIAN PENUH

full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

print(f"Shape telah reindex: {df.shape}")
print(f"Rentang tanggal baru: {df.index.min().date()} s.d. {df.index.max().date()}")

In [ ]:
# TAHAP 4: FORWARD FILL (FFILL) -> HARGA CLOSE HARI TERAKHIR

price_cols = [c for c in df.columns if c.endswith("_close")]
df[price_cols] = df[price_cols].ffill().bfill()

missing_after = df[price_cols].isnull().sum()
print(f"\nJumlah missing value setelah forward fill:\n{missing_after}")
print()


In [ ]:
# # feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df['usdidr_close']

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]
print(df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]].tail(3).to_string())

In [ ]:
# 2. Suruh YData buat "cetak biru" analisisnya
profile = ProfileReport(df, title="Laporan Analisis Data Melzzz")

# 3. Simpan hasilnya jadi halaman web (HTML)!
profile.to_file("laporan_web_kamu.html")

In [ ]:
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
#Setelah mendapatkan dataclean . hapus field antara X dan Y

df['harga_besok_per_gram'] = df['gold_close_idr_per_gram'].shift(-1)
df = df.dropna()
dataclean = df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram", "harga_besok_per_gram"]]
dataclean.tail()

x = dataclean[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]]
y = dataclean["harga_besok_per_gram"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
score = model.score(X_test, y_test)
print(f"R2 Score: {score} ")
print(f"R2 Score: {r2_score(y_test, y_pred)}")
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")